# Reciter TTS — упаковка модели sherpa-onnx (Piper RU / Kokoro) для Android

Скачивает готовую модель из релизов **k2-fsa/sherpa-onnx**, добавляет `model.json` и переупаковывает в ZIP (с сохранением дерева `espeak-ng-data/`) для импорта в приложении.

По умолчанию — **Piper Russian** (нативный русский, реальное время на CPU). Можно выбрать Kokoro (мультиязычный, как 'Orion' в AudiFlo — русский с акцентом).

Не нужен GPU. Просто скачивание + переупаковка.


## Ячейка 1: Выбор модели


In [ ]:
import os, json, tarfile, zipfile, urllib.request, glob

# --- выбери модель ---
# "piper_ru"  : Piper русский (нативный, реальное время)  [рекомендую]
# "kokoro"    : Kokoro multi-lang (как Orion, русский с акцентом)
CHOICE = "piper_ru"

REL = "https://github.com/k2-fsa/sherpa-onnx/releases/download/tts-models"
MODELS = {
  "piper_ru": {
     "url": f"{REL}/vits-piper-ru_RU-dmitri-medium.tar.bz2",
     "arch": "sherpa-vits", "sr": 22050,
     "voices": [{"id":"ru_dmitri","locale":"ru-RU","displayName":"Дмитрий","speakerId":0}],
     "display": "Piper Russian (Дмитрий)",
  },
  "kokoro": {
     "url": f"{REL}/kokoro-int8-multi-lang-v1_0.tar.bz2",
     "arch": "sherpa-kokoro", "sr": 24000,
     # имена дикторов Kokoro (часть); speakerId = индекс в voices.bin
     "voices": [{"id":n,"locale":"en-US","displayName":n,"speakerId":i} for i,n in
                enumerate(["af_heart","af_bella","am_michael","am_adam","bf_emma","am_onyx"])],
     "display": "Kokoro multi-lang (Orion-like)",
  },
}
M = MODELS[CHOICE]
print("выбрано:", M["display"], "->", M["url"])

## Ячейка 2: Скачать + распаковать


In [ ]:
WORK = "/content/sherpa_model"; os.makedirs(WORK, exist_ok=True)
arc = f"{WORK}/model.tar.bz2"
print("качаю..."); urllib.request.urlretrieve(M["url"], arc)
print("распаковка..."); 
with tarfile.open(arc, "r:bz2") as t: t.extractall(WORK)
# найти корневую папку модели (в архиве один верхний каталог)
roots = [d for d in glob.glob(f"{WORK}/*") if os.path.isdir(d)]
ROOT = roots[0]
print("содержимое модели:", sorted(os.listdir(ROOT))[:20])
onnx = [f for f in os.listdir(ROOT) if f.endswith(".onnx")]
print("onnx:", onnx, "| есть espeak-ng-data:", os.path.isdir(f"{ROOT}/espeak-ng-data"),
      "| tokens.txt:", os.path.exists(f"{ROOT}/tokens.txt"))

## Ячейка 3: model.json + сборка ZIP


In [ ]:
onnx_name = sorted([f for f in os.listdir(ROOT) if f.endswith(".onnx")])[0]
# для piper в .onnx.json есть sample_rate/num_speakers
meta_sr = M["sr"]
mj = f"{ROOT}/{onnx_name}.json"
if os.path.exists(mj):
    try: meta_sr = json.load(open(mj)).get("audio",{}).get("sample_rate", M["sr"])
    except Exception: pass

manifest = {
  "schemaVersion": 1, "id": f"sherpa-{CHOICE}", "displayName": M["display"],
  "family": "sherpa", "architecture": M["arch"], "sampleRateHz": int(meta_sr),
  "runtime": "sherpa-onnx",
  "tokenizer": {"type": "espeak-ng", "files": ["tokens.txt"]},
  "files": [{"filename": onnx_name, "role": "TALKER", 
             "sizeMb": round(os.path.getsize(f"{ROOT}/{onnx_name}")/1024**2), "required": True}],
  "voices": M["voices"],
}
json.dump(manifest, open(f"{ROOT}/model.json","w"), ensure_ascii=False, indent=2)
print("model.json:", json.dumps(manifest, ensure_ascii=False)[:300])

# zip: всё содержимое ROOT под models/ (с сохранением подпапок espeak-ng-data/)
zip_path = f"/content/reciter-{CHOICE}-android.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for dp, _, files in os.walk(ROOT):
        for fn in files:
            full = os.path.join(dp, fn)
            rel = os.path.relpath(full, ROOT)
            zf.write(full, f"models/{rel}")
print("архив:", round(os.path.getsize(zip_path)/1024**2,1), "MB ->", zip_path)
try:
    from google.colab import files; files.download(zip_path)
except Exception: pass